# Using the PIC-SURE API in Python

**Search the data dictionary, build a query, export participant-level data.**

---

### What this notebook covers

1. Connect to the NHLBI BioData Catalyst PIC-SURE API with your personal security token
2. Search the **variable dictionary** across every study you are authorized for
3. Read a **concept path** and understand how PIC-SURE organizes dbGaP data
4. Build a query from **clauses** and **clause groups** (`FILTER`, `REQUIRE`, `ANYRECORD`, `AND`/`OR`)
5. Get a **cohort count** before you pull data
6. **Export** participant-level data as a `pandas` DataFrame and write it to disk

### Example with tutorial data (open access)
For the purposes of demonstration and to protect controlled participant-level data, we will be using a tutorial dataset that is synthetic and open access. The notebook uses on of the [BioLINCC Teaching Datasets](https://biolincc.nhlbi.nih.gov/teaching/) available on BDC: the tutorial Framingham Heart Study. 

## 1. The `picsure` Python adapter

The `picsure` package is a thin Python client over the PIC-SURE API. Three groups of calls do almost everything:

| Purpose | Calls |
|---|---|
| Find variables | `session.searchDictionary()`, `session.facets()` |
| Describe a cohort | `picsure.buildClause()`, `picsure.buildClauseGroup()`, `picsure.buildQuery()` |
| Get data out | `session.runQuery()` |

Searches and results come back as **pandas DataFrames**.

Two properties of the adapter worth knowing before you start:

- **Your token carries your dbGaP authorizations.** The dictionary search and the query only ever return studies you are approved for, so two people running this identical notebook can get different results. 
- **Queries return participant-level controlled-access data.** Keep exported DataFrames inside your approved workspace and out of version control.

The R API mirrors these function names, so a query translates between the two languages almost line for line.

Source: <https://github.com/hms-dbmi/pic-sure-python-adapter-hpds>

## 2. Environment set-up

**Prerequisites:** Python 3.10 or later, and `pip`.
Run the install cell below once.

In [ ]:
%pip install picsure

In [ ]:
import picsure
import pandas as pd

## 3. Your security token

PIC-SURE authenticates you with a **personal security token**. It encodes who you are and, by extension, which studies you can see.

To get one:

1. Open <https://picsure.biodatacatalyst.nhlbi.nih.gov/> and log in via **RAS**
2. Click the **Prepare for Analysis** tab: <https://picsure.biodatacatalyst.nhlbi.nih.gov/analyze/api>
3. Click **Copy** in the Personalized Access Token box
4. Save it to a file named `token.txt` next to this notebook

> **Your token is personal — treat it like a password.** Do not paste it into a shared notebook, do not commit `token.txt`.

In [ ]:
token_file = "token.txt"

try:
    with open(token_file, "r") as f:
        my_token = f.read().strip()
    print(f"Token loaded from {token_file}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"No {token_file} found in this directory."
    )

### Connecting

`picsure.connect()` takes the platform you want and your token. `Platform.BDC_AUTHORIZED` indicates that BDC PIC-SURE is the platform to connect to.

In [ ]:
session = picsure.connect(picsure.Platform.BDC_AUTHORIZED, my_token)

## 4. The variable dictionary

Everything in PIC-SURE starts with the dictionary. `session.searchDictionary(term)` does a free-text search across variable names, descriptions, dataset names, and study metadata, and hands back a **pandas DataFrame** — one row per variable.

Let's search broadly first, then narrow to our study. We will use the search term `framingham`.

In [ ]:
search_df = session.searchDictionary("framingham")
search_df.head()

### Anatomy of a result row

The columns you will use most:

| Column | What it holds |
|---|---|
| `conceptPath` | **The identifier you query with.** This describes the hierarchy associated with the variable. |
| `name` | Accession number associated with the variable (e.g. `phv1234`). Note when not available it is the same as `display` |
| `display` | Encoded variable name from the data sumbitters (e.g. `BMI1`) |
| `description` | Human-readable description of the variable (e.g. `Body mass index at visit 1`) |
| `dataType` | Describes the type of variable, either categorical or continuous |
| `studyId` | `phs` accession of the study or, if not available, other unique identifier|
| `values` | For categorical variables: the allowed values |
| `min` / `max` | For continuous variables: the observed range |
| `allowFiltering` | Boolean, whether filtering is allowed on this variable |
| `studyAcronym` | Abbreviation of the associated study (e.g. `FHS`) |

Concept paths for controlled access data are generally formatted as `\phs\pht\phv\variable name\`. There are some exceptions to this, particularly with studies not indexed by dbGaP. 

Note that the tutorial datasets are unique, which concept paths formatted as `\study ID\variable name\`.

In [ ]:
study_id = "tutorial-biolincc_framingham"

# Narrow to our study of interest.
study_df = search_df[search_df.studyId == study_id]

study_df.head()

### Filtering the search with facets

Rather than searching everything and subsetting the DataFrame, you can push the study filter into the search itself with a **facet**. This is the better habit for large searches — less data over the wire.

In [ ]:
facets = session.facets()                  # start an empty facet object
facets.add("dataset_id", study_id)         # restrict to our study
facets.view()                              # confirm what is applied

In [ ]:
# New search, scoped by the facet
bmi_search = session.searchDictionary(term="body mass index", facets=facets)
bmi_search.head(10)

## 5. Building a query

There are three parts of a query:

1. **Clauses** — one filtering condition on one (or more) concept paths. Built with `picsure.buildClause()`.
2. **Clause groups** — clauses joined by `AND` or `OR`. Built with `picsure.buildClauseGroup()`. Groups can nest inside other groups, which is how you express things like *(A or B) and C*.
3. **The query** — `picsure.buildQuery()`, which takes your filter logic plus any extra variables you want returned but do not want to filter on.

### The clause types

| `PhenotypicFilterType` | Arguments | Effect |
|---|---|---|
| `FILTER` | concept path, `categories=` **or** `min=`/`max=` | Keep participants matching the criteria |
| `REQUIRE` | concept path(s) | Keep participants with a non-null value for **all** listed paths |
| `ANYRECORD` | concept path(s) | Keep participants with a non-null value for **at least one** listed path |

Note that passing several paths to a single `REQUIRE` clause combines them with OR, not AND. If you need *both* variables present, build one `REQUIRE` clause per path and join them with an `AND` clause group. (This is called out explicitly below.)

### Our example cohort

We will select the tutorial Framingham participants who:
- are **current smokers**, which we will define with two variables:
    - whether they were **current cigarette smokers** (categorical `FILTER`), **or**
    - if they reported **smoking some cigarettes per day** (categorical `FILTER`)
**and**
- have **age above 50** (continuous `FILTER`), **and**
- have a recorded **body mass index** (`REQUIRE`)

and we will additionally **return** variables related to hypertension without filtering on it. 

### 5a. Categorical filtering - current cigarette smokers

First find the variables related to cigarette smoking, then look at what values are available for filtering.

In [ ]:
cigarette_search = session.searchDictionary(term="cigarette", facets=facets)
cigarette_search.head()

In [ ]:
# Let's look at the values available for "Number of cigarettes smoked each day"
number_cigs_values = cigarette_search.loc[cigarette_search.name == "CIGPDAY","values"].iloc[0]
print(number_cigs_values)

# We want to include all values but "Not current smoker"
filtered_number_cigs_values = [val for val in number_cigs_values if val != "Not current smoker"]
print(filtered_number_cigs_values)

# We will also save the concept path to use later
number_cigs_path = cigarette_search.loc[cigarette_search.name == "CIGPDAY","conceptPath"].iloc[0]
print(number_cigs_path)

In [ ]:
# Let's look at the values available for "Current cigarette smoking at exam"
current_smoking_values = cigarette_search.loc[cigarette_search.name == "CURSMOKE","values"].iloc[0]
print(current_smoking_values)

# We only want one value "Current smoker", which is information we will use later when applying the filter

# We will also save the concept path to use later
current_smoking_path = cigarette_search.loc[cigarette_search.name == "CURSMOKE","conceptPath"].iloc[0]
print(current_smoking_path)

In [ ]:
# Next we will create two filter Clauses, then combine them into a ClauseGroup.

# First filter for number of cigarettes
number_cigs_clause = picsure.buildClause(
    number_cigs_path,
    picsure.PhenotypicFilterType.FILTER,
    categories=filtered_number_cigs_values, # `categories` accepts a list or string, here we provide a list of values
)
print(number_cigs_clause)

# Second filter for current smokers
current_smoking_clause = picsure.buildClause(
    current_smoking_path,
    picsure.PhenotypicFilterType.FILTER, 
    categories="Current smoker", # `categories` accepts a list or string, here we provide a string
)
print(current_smoking_clause)

In [ ]:
# Now we will create a ClauseGroup to combine the two filters. We will use the "OR" operator, meaning that we will include participants who meet either of the two criteria.
smoke_clause_group = picsure.buildClauseGroup(
    [number_cigs_clause, current_smoking_clause],
    operator=picsure.GroupOperator.OR
)
print(smoke_clause_group)

### 5b. Continuous filtering - age above 50

Same idea, but filtering using `min=` / `max=` instead of a list of values. Either bound can be omitted for an open-ended range. Check the `min`/`max` columns first so your threshold is actually inside the observed range.

In [ ]:
age_search = session.searchDictionary(term="age", facets=facets)
print(age_search.head())

# There is only one age variable returned with min=32 and max=81. Let's save the concept path and build a Clause to filter on the variable.
age_path = age_search.loc[age_search.name == "AGE","conceptPath"].iloc[0]
print(age_path)

# Now we will build a Clause to filter on the age variable.
age_clause = picsure.buildClause(
    age_path,
    picsure.PhenotypicFilterType.FILTER,
    min = 50 # Excluding the `max` argument will include all participants with age >= 50
)
print(age_clause)


### 5c. `REQUIRE` — "must have a measurement"

Often you do not care what the value is, only that it exists.

Note the two separate clauses. One `REQUIRE` clause holding both paths would mean *variable 1 OR variable 2*; two clauses joined with `AND` means *variable 1 AND variable 2*. Here, we will only add one variable to select measurements for BMI. 

In [ ]:
bmi_search = session.searchDictionary(term="body mass index", facets=facets)
bmi_search.head()

In [ ]:
# Save the concept path to build a clause 
bmi_path = bmi_search["conceptPath"].tolist()
print(bmi_path)

# Create a clause to require the BMI measurement
bmi_clause = picsure.buildClause(
    bmi_path,
    picsure.PhenotypicFilterType.REQUIRE
)
print(bmi_clause)

### 5d. Combining into a clause group

Now, we can put our clauses and clause groups together to finalize our cohort. As a reminder, we will be creating a query to select **smokers `AND` age above 50 `AND` those with BMI measurements**. 

In [ ]:
cohort_filter = picsure.buildClauseGroup(
    [smoke_clause_group, age_clause, bmi_clause],
    operator=picsure.GroupOperator.AND,
)
cohort_filter

### 5e. Building the query

`includeConcepts` is the "return this column but do not filter on it" export option. Variables you filtered on are automatically returned, so this is an optional argument for additional variables. Let's include variables related to hypertension.

In [ ]:
hyperten_search = session.searchDictionary(term="hypertension", facets=facets)
hyperten_search.head()

In [ ]:
# We will save those concept paths to include in the final query
hyperten_paths = hyperten_search["conceptPath"].tolist()
print(hyperten_paths)

In [ ]:
demo_query = picsure.buildQuery(
    phenotypicFilter=cohort_filter,
    includeConcepts=hyperten_paths,
)
demo_query

## 6. Running the query

Two query types matter here:

- `QueryType.COUNT` — how many participants match. Cheap, fast, no data leaves the server.
- `QueryType.PARTICIPANT` — the actual participant-level DataFrame.

In [ ]:
cohort_count = session.runQuery(demo_query, type=picsure.QueryType.COUNT)
print(cohort_count)

In [ ]:
results = session.runQuery(demo_query, type=picsure.QueryType.PARTICIPANT)
print(f"Exported shape: {results.shape}")
results.head()

### Reading the output

Each **row** is a participant; each **column** is a concept path, plus automatically-included columns such as `patient_id`.

**A cell can hold multiple values separated by tabs (`\t`)**. This may be because a participant has repeated measurements associated with a single variable. Whether this is the case depends on the study design and data collection methods.

## 7. Bonus: start in the UI, finish in Python

Most people find a cohort faster by clicking through the PIC-SURE web UI than by writing clauses. You do not have to choose. In the UI, build your cohort and click **Copy Query ID**, then bring it here:

In [ ]:
queryID = "paste-the-query-id-here"

# Run it directly
df_from_ui = session.runQueryByID(queryID, type=picsure.QueryType.PARTICIPANT)

# Or load it as an object you can inspect and modify programmatically
query = session.loadQueryByID(queryID)
query

----

## 8. Lab use case: Export based on tables (phts)

What if you want to export an entire table, or pht? Thi sis done by searching for the pht of interest, saving all associated concept paths, and building a query for export. 

> **WARNING:** The cells below are related to participant-level data. Please execute them according to your **Data Use Agreement**.

Let's use the pht007777. Using the [dbGaP search](https://dbgap.ncbi.nlm.nih.gov/beta/study/phs000007.v35.p16/dataset/pht007777.v3.p16/#phenotype-datasets), we can see that this dataset is related to Frequently Used Cardiovascular Risk Factors for the Original Cohort for Exam 1 - Exam 32.

In [ ]:
pht = "pht007777" # Change to pht of interest

pht_search = session.searchDictionary(term=pht)
pht_search.head()

In [ ]:
# Ensure that all concept paths contain the pht of interest
pht_search = pht_search[pht_search.conceptPath.str.contains(pht)]

# Save concept paths for query building
pht_paths = pht_search.conceptPath.tolist()

In [ ]:
pht_clause = picsure.buildClause(
    pht_paths,
    picsure.PhenotypicFilterType.ANYRECORD
)

pht_query = picsure.buildQuery(
    phenotypicFilter=pht_clause
)


In [ ]:
pht_count = session.runQuery(pht_query, type=picsure.QueryType.COUNT)
print(pht_count)

In [ ]:
# EXPORTS PARTICIPANT-LEVEL DATA!!!

#pht_results = session.runQuery(pht_query, type=picsure.QueryType.PARTICIPANT)
#print(f"Exported shape: {pht_results.shape}")  
#pht_results.head()